
# Tutorial 2 — Finding papers to extract from

Before you can extract anything, you need papers. This tutorial works through
[LeMat-Synth-Papers](https://huggingface.co/datasets/LeMaterial/LeMat-Synth-Papers)
— the corpus of ~81k open-access materials science papers (full text, not just
abstracts) that feeds the extraction pipeline — and narrows it down to the
handful of papers you actually care about.

## What you'll learn

1. How the corpus is organised (three sources, one schema)
2. How to filter by **category**, the source-specific subject labels
3. How to filter by **keyword** with whole-word matching — and why naive
   substring matching quietly ruins short-token searches like `Tc`
4. How to swap in an **LLM relevance filter** when keywords are not selective
   enough
5. How to export the resulting paper list for the extraction tutorials

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- Access to the gated dataset — see Step 0
- **Runtime:** 10–30 min, most of it downloading. **Cost:** free unless you run
  the optional LLM filter.


## Step 0 — Dataset access and your `.env` file

`LeMat-Synth-Papers` is **gated**: request access once on the Hub, then
authenticate locally. Either option works:

**Option A — log in once, globally:**

```bash
hf auth login          # paste a token from huggingface.co/settings/tokens
```

**Option B — put the token in `.env`,** the same file this project uses for
every other credential. From the repository root:

```bash
cp .env.example .env
```

then add:

```
HF_TOKEN=hf_...
```

`.env` is git-ignored so the token is never committed, and everything in this
project — CLI, notebooks, scripts — reads credentials from it rather than from
hard-coded strings.

> **Heads-up on size.** The `full` config is large: `arxiv` ≈ 1.8 GB of
> parquet, `chemrxiv` ≈ 4.0 GB, `omg24` ≈ 4.3 GB, cached under
> `~/.cache/huggingface` on first use. To try the notebook out cheaply, load
> the `superconductor_keywords_and_LLM` config (1.4k rows, ~37 MB) instead —
> the loading cell below shows how.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# find_dotenv walks up from the working directory, so this works whether you
# started Jupyter at the repo root or inside this folder.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

print(f".env loaded from: {env_path or 'NOT FOUND'}")
if os.getenv("HF_TOKEN"):
    print(f"HF_TOKEN found ({len(os.getenv('HF_TOKEN'))} chars)")
else:
    print("No HF_TOKEN in .env - relying on `hf auth login` instead.")

from huggingface_hub import whoami

try:
    print(f"Authenticated as: {whoami()['name']}")
except Exception as exc:
    raise RuntimeError(
        "Not authenticated with the HuggingFace Hub. Run `hf auth login`, or "
        "add HF_TOKEN=... to your .env at the repository root."
    ) from exc

In [ ]:
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 120)

# Load the three raw splits from HuggingFace (gated: run `hf auth login` first).
# Downloads ~10 GB of parquet in total and caches it under ~/.cache/huggingface.
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",
    split=None,
    token=True,
)

# Smaller alternatives for a quick look (same column layout):
# dataset = load_dataset(
#     "LeMaterial/LeMat-Synth-Papers", "superconductor_keywords_and_LLM",
#     split=None, token=True,
# )

print("Available splits:")
for name, ds in dataset.items():
    print(f"  {name}: {len(ds)} papers")
print(f"\nColumns: {dataset[next(iter(dataset.keys()))].column_names}")

---
## Step 1 — Choose split(s)

Pick one or more raw sources to combine.

In [ ]:
from datasets import concatenate_datasets

# Pick one or more splits: "arxiv", "omg24", "chemrxiv"
SPLITS = ["arxiv", "omg24"]  # e.g. ["arxiv", "omg24"] to combine two sources


ds = concatenate_datasets([dataset[s] for s in SPLITS])
print(f"Selected splits {SPLITS}: {len(ds)} papers total")

---
## Step 2 — Explore categories

The `categories` column is a string. Individual categories are separated by `,` or `;`.

- **arxiv**: codes like `cond-mat.mtrl-sci`, `physics.chem-ph`
- **chemrxiv**: names like `Solid State Chemistry`, `Surface`
- **omg24**: Semantic Scholar field-of-study labels

In [ ]:
def parse_categories(cat_string):
    """Split a categories string by comma and semicolon into a list."""
    if cat_string is None:
        return []
    parts = []
    for part in cat_string.split(","):
        for sub_part in part.split(";"):
            cleaned = sub_part.strip()
            if cleaned:
                parts.append(cleaned)
    return parts


# Flatten all categories across the selected split(s)
all_cats = []
for cat_string in ds["categories"]:
    all_cats.extend(parse_categories(cat_string))

cat_counts = pd.Series(all_cats).value_counts()
print(f"{len(cat_counts)} unique categories across {len(ds)} papers\n")
print("Top 20 categories:")
print(cat_counts.head(20).to_string())

## Step 3 — Filter by categories

Keep only papers whose `categories` string contains at least one match.  
Uses substring matching (case-insensitive).  
Set `CATEGORY_FILTER = []` to skip and keep all papers.

In [ ]:
# Categories to keep (substring match, case-insensitive)

# Examples:
#   arxiv:    ["cond-mat"]
#   chemrxiv: ["Solid State Chemistry", "Surface"]
CATEGORY_FILTER = ["Superconductor"]  # empty = keep all papers


def filter_by_category(ds, category_filter):
    """Keep papers whose categories string contains any of the filter terms."""
    if not category_filter:
        return ds
    cat_lower = [c.lower() for c in category_filter]

    def has_category(example):
        cats = example["categories"]
        if cats is None:
            return False
        cats_lower = cats.lower()
        return any(cf in cats_lower for cf in cat_lower)

    return ds.filter(has_category)


ds_cat = filter_by_category(ds, CATEGORY_FILTER)

if CATEGORY_FILTER:
    print(
        f"Category filter {CATEGORY_FILTER}: {len(ds_cat)} / {len(ds)} papers"
    )
else:
    print(f"No category filter applied: {len(ds_cat)} papers")

---
## Step 4 — Keyword filter

### Search configuration

These settings are used by **both** filtering options below, and by the results
section — run this cell whichever option you pick.

The helper compiles each keyword into a **whole-word** pattern. That detail
matters more than it looks: a plain `"tc" in text.lower()` test also fires on
*ba**tc**h*, *ma**tc**h* and *swi**tc**h*, which turns a superconductivity
search into noise. Word boundaries keep `Tc` meaning the critical temperature.

In [ ]:
import re

# Column to search in
TEXT_COLUMN = "abstract"

# Include keywords — papers matching ANY of these are kept
INCLUDE_KEYWORDS = ["resistivity", "resistance", "critical temperature", "Tc"]

EXCLUDE_KEYWORDS = ["semiconductor"]

# How many papers to display
NUM_DISPLAY = 20


def keyword_pattern(keyword):
    """Compile a case-insensitive whole-word pattern for a keyword.

    Word boundaries matter for short tokens: a plain substring test for "Tc"
    also matches "batch", "match" and "switch".
    """
    return re.compile(rf"\b{re.escape(keyword)}\b", re.IGNORECASE)

### Option 1: Direct keyword matching (default)

Papers matching **any** include keyword (and **none** of the exclude keywords) are kept.

In [ ]:
def keyword_filter(ds, text_column, include_kws, exclude_kws=None):
    """Filter HuggingFace Dataset by include/exclude keywords."""
    include_patterns = [keyword_pattern(kw) for kw in include_kws]
    exclude_patterns = [keyword_pattern(kw) for kw in exclude_kws or []]

    def matches_any(example, patterns):
        if example[text_column] is None:
            return False
        return any(p.search(example[text_column]) for p in patterns)

    filtered = ds.filter(lambda x: matches_any(x, include_patterns))

    if exclude_patterns:
        filtered = filtered.filter(
            lambda x: not matches_any(x, exclude_patterns)
        )

    return filtered


ds_filtered = keyword_filter(
    ds_cat, TEXT_COLUMN, INCLUDE_KEYWORDS, EXCLUDE_KEYWORDS
)
print(f"Keyword filter: {len(ds_filtered)} / {len(ds_cat)} papers")

### Option 2: LLM-based keyword filter (commented out)

Instead of keyword matching, use an LLM to decide whether a paper is relevant.  
Run the shared search configuration cell above first — the results section uses it.  
Requires a running **vLLM** endpoint. Uncomment the cell below to use it.

Two ways to reach a model, both through the OpenAI-compatible client:

| | Endpoint | Key | Notes |
|---|---|---|---|
| **Local vLLM** | `http://localhost:8000/v1` | not needed | free, needs a GPU and a served model |
| **OpenRouter** | `https://openrouter.ai/api/v1` | `OPENROUTER_API_KEY` | no infrastructure; costs per paper, so filter by category and keyword *first* |

**Prerequisites:**
- `pip install openai` (plus `transformers` only for the local token-counting path)
- For OpenRouter: `OPENROUTER_API_KEY` in your `.env` at the repository root

In [ ]:
# # --- Option 2: LLM-based filtering ---
# # Uncomment this entire cell to use LLM filtering instead of direct keywords.
# # Run this instead of the Option 1 filter cell above (the shared
# # search configuration cell is still required).
#
# import openai
# from tqdm import tqdm
#
# # --- Pick one -------------------------------------------------------------
# USE_OPENROUTER = False
#
# if USE_OPENROUTER:
#     # Any model id from https://openrouter.ai/models
#     ENDPOINT = "https://openrouter.ai/api/v1"
#     API_KEY = os.environ["OPENROUTER_API_KEY"]
#     MODEL_NAME = "mistralai/ministral-3-14b-instruct"
# else:
#     ENDPOINT = "http://localhost:8000/v1"
#     API_KEY = "not-needed"
#     MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
#
# MAX_MODEL_LEN = 50000
#
# # Edit this prompt to match your use case
# LLM_PROMPT = """You are provided with a scientific materials paper.
# Read the paper carefully and determine if it is relevant to the topic
# of heterogeneous catalysis with temperature-dependent performance data.
#
# Answer with only yes or no.
# If you are not sure, answer with no.
#
# Paper: {paper_text}
# Question: Is this paper relevant?
# Answer:"""
#
# client = openai.OpenAI(base_url=ENDPOINT, api_key=API_KEY)
#
#
# def ask_llm(text):
#     """Send paper text to the LLM and return True if the paper is relevant."""
#     # Rough character budget instead of a tokenizer: ~4 chars per token keeps
#     # this dependency-free and works for either endpoint.
#     text = text[: MAX_MODEL_LEN * 4]
#     message = LLM_PROMPT.format(paper_text=text)
#     try:
#         response = client.chat.completions.create(
#             model=MODEL_NAME,
#             messages=[{"role": "user", "content": message}],
#             temperature=0,
#             max_tokens=100,
#         )
#         answer = response.choices[0].message.content.strip().lower()
#         return "yes" in answer
#     except Exception as e:
#         print(f"LLM call failed: {e}")
#         return False
#
#
# # Run LLM filter on the category-filtered dataset (ds_cat)
# llm_results = []
# for paper in tqdm(ds_cat, desc="LLM filtering"):
#     text = paper.get("text_paper") or paper.get("abstract") or ""
#     llm_results.append(ask_llm(text))
#
# ds_cat_with_labels = ds_cat.add_column("llm_relevant", llm_results)
# ds_filtered = ds_cat_with_labels.filter(lambda x: x["llm_relevant"])
# print(f"LLM filter: {len(ds_filtered)} / {len(ds_cat)} papers")

---
## Step 5 — Results

### Per-keyword match counts

In [ ]:
# Per-keyword counts on the filtered dataset
def count_keyword_matches(ds, text_column, keywords):
    counts = {}
    for kw in keywords:
        pattern = keyword_pattern(kw)

        def has_keyword(example, pattern=pattern):
            if example[text_column] is None:
                return False
            return bool(pattern.search(example[text_column]))

        counts[kw] = len(ds.filter(has_keyword))
    return counts


keyword_counts = count_keyword_matches(
    ds_filtered, TEXT_COLUMN, INCLUDE_KEYWORDS
)
print("Per-keyword match counts:")
for kw, count in sorted(keyword_counts.items(), key=lambda x: -x[1]):
    print(f"  {kw}: {count}")
print(f"\nTotal papers: {len(ds_filtered)}")

### Browse results

In [ ]:
DISPLAY_COLUMNS = ["id", "title", "categories", "abstract", "pdf_url"]

cols = [c for c in DISPLAY_COLUMNS if c in ds_filtered.column_names]
ds_filtered.to_pandas()[cols].head(NUM_DISPLAY)

### Export (optional)

In [ ]:
# df = ds_filtered.to_pandas()
# df.to_csv("filtered_papers.csv", index=False)
# df.to_pickle("filtered_papers.pkl")

## What's next

You now have a filtered set of papers. To turn them into structured data:

- **[Tutorial 1 — Explore the LeMat-Synth dataset](01_explore_the_lemat_synth_dataset.ipynb)**:
  check whether the papers you selected have *already* been extracted before
  spending API credits on them.
- **[Tutorial 3 — Batch extraction with the CLI](03_batch_extraction_with_the_cli.ipynb)**:
  point `lemat-synth batch` at a folder of papers and let it run.
- **[Tutorial 4 — Synthesis + performance from a paper](04_extracting_synthesis_and_performance.ipynb)**:
  take one paper from PDF to a validated `GeneralSynthesisOntology` object, one
  pipeline stage at a time.

To fetch the PDFs themselves, the `pdf_url` column is the starting point;
`examples/scripts/data_curation/` holds the download and curation scripts used
to build the corpus.
